# Day 042 Project: Query a Database

## What You're Building

A set of SQL queries against the retail sales database demonstrating SELECT, WHERE, GROUP BY, ORDER BY, and INNER JOIN.

**Deliverable:** All five queries run and `_run_project_checks()` passes.

## Project Requirements

1. Create an in-memory database with `setup_db(conn)`
2. `q1` — all Accessories orders ordered by revenue desc
3. `q2` — revenue summary grouped by region
4. `q3` — top product by total revenue (use GROUP BY + ORDER BY + LIMIT 1)
5. `q4` — JOIN summary of region × category
6. `q5` — query into pandas with `pd.read_sql_query`

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sqlite3
import pandas as pd


import sqlite3

def setup_db(conn):
    cur = conn.cursor()
    cur.execute('''
        CREATE TABLE IF NOT EXISTS orders (
            order_id  INTEGER PRIMARY KEY,
            product   TEXT,
            category  TEXT,
            region    TEXT,
            price     REAL,
            quantity  INTEGER,
            revenue   REAL
        )''')
    cur.execute('''
        CREATE TABLE IF NOT EXISTS products (
            product    TEXT PRIMARY KEY,
            category   TEXT,
            unit_price REAL
        )''')
    rows = [
        (1,'Widget','Electronics','North',25.0,10,250.0),
        (2,'Gadget','Electronics','South',150.0,3,450.0),
        (3,'Widget','Electronics','South',25.0,5,125.0),
        (4,'Doohickey','Accessories','East',8.0,50,400.0),
        (5,'Gadget','Electronics','East',150.0,7,1050.0),
        (6,'Widget','Electronics','East',25.0,4,100.0),
        (7,'Doohickey','Accessories','North',8.0,20,160.0),
        (8,'Gadget','Electronics','North',150.0,2,300.0),
        (9,'Widget','Electronics','West',25.0,6,150.0),
        (10,'Doohickey','Accessories','South',8.0,15,120.0),
        (11,'Thingamajig','Accessories','North',200.0,1,200.0),
        (12,'Thingamajig','Accessories','East',200.0,4,800.0),
    ]
    cur.executemany(
        'INSERT OR IGNORE INTO orders VALUES (?,?,?,?,?,?,?)', rows
    )
    products = [
        ('Widget','Electronics',25.0),
        ('Gadget','Electronics',150.0),
        ('Doohickey','Accessories',8.0),
        ('Thingamajig','Accessories',200.0),
    ]
    cur.executemany(
        'INSERT OR IGNORE INTO products VALUES (?,?,?)', products
    )
    conn.commit()


def run_query(conn, sql, params=()):
    cur = conn.cursor()
    cur.execute(sql, params)
    cols = [col[0] for col in cur.description]
    return [dict(zip(cols, row)) for row in cur.fetchall()]


def filter_orders(conn, region=None, category=None, min_revenue=None):
    conditions = []
    params = []
    if region is not None:
        conditions.append('region = ?')
        params.append(region)
    if category is not None:
        conditions.append('category = ?')
        params.append(category)
    if min_revenue is not None:
        conditions.append('revenue >= ?')
        params.append(min_revenue)
    where = ('WHERE ' + ' AND '.join(conditions)) if conditions else ''
    sql = f'SELECT * FROM orders {where} ORDER BY order_id'
    return run_query(conn, sql, tuple(params))


def group_revenue(conn, group_col):
    sql = (
        f'SELECT {group_col}, SUM(revenue) AS total, '
        'COUNT(*) AS orders, ROUND(AVG(revenue), 2) AS avg_revenue '
        f'FROM orders GROUP BY {group_col} ORDER BY total DESC'
    )
    return run_query(conn, sql)


def join_summary(conn):
    sql = (
        'SELECT o.region, p.category, '
        'SUM(o.revenue) AS total_revenue, COUNT(*) AS order_count '
        'FROM orders o '
        'INNER JOIN products p ON o.product = p.product '
        'GROUP BY o.region, p.category '
        'ORDER BY total_revenue DESC'
    )
    return run_query(conn, sql)


conn = sqlite3.connect(':memory:')
setup_db(conn)
print('Database ready.')

## Your Queries

In [ ]:
# q1: all Accessories orders, most expensive first
# TODO: q1 = filter_orders(conn, category='Accessories')
# TODO: q1 = sorted(q1, key=lambda r: r['revenue'], reverse=True)
# TODO: print(f'q1: {len(q1)} Accessories orders')

# q2: revenue grouped by region
# TODO: q2 = group_revenue(conn, 'region')
# TODO: print('q2:', [(r['region'], r['total']) for r in q2])

# q3: single top product by revenue (raw SQL with LIMIT)
# TODO: q3 = run_query(conn,
# TODO:     'SELECT product, SUM(revenue) AS total FROM orders'
# TODO:     ' GROUP BY product ORDER BY total DESC LIMIT 1')
# TODO: print('q3 top product:', q3[0])

# q4: JOIN summary
# TODO: q4 = join_summary(conn)
# TODO: print(f'q4: {len(q4)} region/category combos')

# q5: query into pandas
# TODO: q5_df = pd.read_sql_query('SELECT * FROM orders', conn)
# TODO: print(f'q5 DataFrame shape: {q5_df.shape}')

## Project Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: q1 contains only Accessories, >= 5 rows
    try:
        assert 'q1' in globals(), 'q1 not defined'
        assert all(r['category'] == 'Accessories' for r in q1)
        assert len(q1) == 5, f'expected 5 Accessories rows, got {len(q1)}'
        passed += 1; print(f'\u2705 Check 1: q1 has {len(q1)} Accessories orders')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: q2 has 4 regions with correct totals
    try:
        assert 'q2' in globals(), 'q2 not defined'
        assert len(q2) == 4, f'expected 4 regions, got {len(q2)}'
        totals = {r['region']: r['total'] for r in q2}
        assert abs(sum(totals.values()) - 4105.0) < 0.01
        passed += 1; print(f'\u2705 Check 2: q2 sums to 4105.0 across 4 regions')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: q3 identifies Gadget as top product
    try:
        assert 'q3' in globals(), 'q3 not defined'
        assert q3[0]['product'] == 'Gadget', \
            f'expected Gadget, got {q3[0]["product"]}'
        assert abs(q3[0]['total'] - 1800.0) < 0.01
        passed += 1; print(f'\u2705 Check 3: q3 top product = Gadget (1800.0)')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: q4 join summary has 7 rows
    try:
        assert 'q4' in globals(), 'q4 not defined'
        assert len(q4) == 7, f'expected 7 join rows, got {len(q4)}'
        assert 'total_revenue' in q4[0]
        passed += 1; print(f'\u2705 Check 4: q4 join summary has 7 rows')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: q5_df is a pandas DataFrame with 12 rows
    try:
        import pandas as pd
        assert 'q5_df' in globals(), 'q5_df not defined'
        assert isinstance(q5_df, pd.DataFrame)
        assert q5_df.shape[0] == 12, f'expected 12 rows, got {q5_df.shape[0]}'
        passed += 1; print(f'\u2705 Check 5: q5_df is DataFrame {q5_df.shape}')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Write a query with `HAVING SUM(revenue) > 500` to filter groups
- Use `LEFT JOIN` instead of `INNER JOIN` and observe which rows change
- Write a subquery: `SELECT * FROM orders WHERE revenue > (SELECT AVG(revenue) FROM orders)`
- On Day 43 you will build `ask_sql(conn, question)` — natural language to SQL generation
  using the same code-generation pattern from Day 41